# LoRA Quick Run — Qwen2-VL-2B on VQA-RAD

**STAT GR5293 GenAI Course Project — Member 1, Week 4**

This notebook runs the **LoRA fine-tuning quick run**: a sub-30-minute end-to-end LoRA training + evaluation on a 200-example subset of VQA-RAD. It validates that the entire training pipeline works on a Colab T4 GPU before committing to a multi-hour full run.

Members 2 (LoRA full / QLoRA) and 3 (DoRA) will extend this notebook by changing only the YAML config, not the code.

## What this notebook does

1. Verifies a GPU is available
2. Installs dependencies (transformers, peft, bitsandbytes, etc.)
3. Trains a LoRA adapter on 200 VQA-RAD train examples (1 epoch)
4. Saves the adapter to `checkpoints/lora_quick/`
5. Evaluates the fine-tuned model on the full 451-example test set
6. Compares LoRA results to the zero-shot baseline (paired statistical tests)
7. Produces a side-by-side metrics table

## Expected runtime

On a free-tier Colab T4 GPU:
* Dependencies: ~3 min
* Model download (Qwen2-VL-2B, ~4 GB): ~3 min (skipped if already cached)
* LoRA training (200 examples, 1 epoch, rank 8): ~10–15 min
* Test-set evaluation (451 examples): ~15–20 min
* **Total: ~30–40 min**

## Prerequisites

1. **Run `02_baseline_zeroshot.ipynb` first.** This notebook compares against the baseline's saved `per_example_scores.json`, so the baseline must already exist at `results/baseline/per_example_scores.json`.
2. **Runtime → Change runtime type → T4 GPU → Save**.

## 1. Environment check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Switch to T4 in Runtime settings.'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Torch: {torch.__version__}')

In [ ]:
%pip install -q --upgrade \
    "transformers>=4.45.0,<4.50.0" \
    "accelerate>=0.34.0" \
    "datasets>=3.0.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.44.0" \
    "qwen-vl-utils>=0.0.8" \
    "rouge-score>=0.1.2"

In [ ]:
import os, sys
USE_DRIVE = True  # Cache the 4 GB model weights on Drive across sessions
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    HF_CACHE = '/content/drive/MyDrive/peft-vqa-cache'
    os.makedirs(HF_CACHE, exist_ok=True)
else:
    HF_CACHE = None

PROJECT_DIR = '/content/peft-medical-vqa'
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(f'{PROJECT_DIR} not found. Upload + unzip the project first.')
%cd {PROJECT_DIR}
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

## 2. Train the LoRA adapter

We use the YAML config at `configs/lora_quick.yaml`:

```yaml
lora_r: 8
lora_alpha: 16
train_max_examples: 200
num_epochs: 1
gradient_checkpointing: true
```

In [ ]:
from src.utils.seed import set_global_seed
from src.training.train_lora import LoRATrainingConfig, train_lora

set_global_seed(42)
cfg = LoRATrainingConfig.from_yaml('configs/lora_quick.yaml')
if HF_CACHE:
    cfg.cache_dir = HF_CACHE
print('Training config:')
for k, v in cfg.__dict__.items():
    print(f'  {k}: {v}')

In [ ]:
metrics = train_lora(cfg)
print('\n✅ LoRA quick run complete.')

## 3. Inspect training metrics

In [ ]:
import json
with open('checkpoints/lora_quick/training_metrics.json') as f:
    m = json.load(f)

print('Method            :', m['method'])
print('Trainable params  :', f"{m['params']['trainable']:,} ({m['params']['trainable_pct']:.4f}% of total)")
print('Total params      :', f"{m['params']['total']:,}")
print('Train time (s)    :', m['training']['total_seconds'])
print('Epochs            :', m['training']['n_epochs'])
print('Mean s / epoch    :', m['training']['mean_epoch_seconds'])
print('Peak GPU (alloc)  :', f"{m['peak_gpu_memory_gb']:.2f} GB")
print('Peak GPU (reserved):', f"{m['peak_gpu_reserved_gb']:.2f} GB")

In [ ]:
# Plot the loss curve
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
loss_df = pd.DataFrame(m['training']['loss_curve'])
if not loss_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(loss_df['step'], loss_df['loss'], marker='.', linewidth=1.5)
    ax.set_xlabel('Optimizer step')
    ax.set_ylabel('Training loss')
    ax.set_title(f'LoRA quick-run loss curve (r={cfg.lora_r}, n_train={cfg.train_max_examples})')
    plt.tight_layout()
    fig.savefig('checkpoints/lora_quick/loss_curve.png', dpi=120)
    plt.show()
else:
    print('No logging steps recorded — increase logging_steps in YAML.')

## 4. Evaluation results

These come from the full 451-example test split. Compared to the zero-shot baseline:

In [ ]:
ev = m['evaluation']

def fmt_ci(point, ci):
    if ci is None: return f'{point:.4f}'
    return f"{point:.4f} [{ci['lower']:.4f}, {ci['upper']:.4f}]"

print('LoRA quick-run evaluation on VQA-RAD test (n=451)')
print('-' * 60)
print(f"  Closed-ended (n={ev['closed']['n']}):")
print(f"    Exact Match : {fmt_ci(ev['closed']['exact_match'], ev['closed'].get('ci95'))}")
print(f"  Open-ended (n={ev['open']['n']}):")
print(f"    BLEU-1      : {ev['open']['bleu1']:.4f}")
print(f"    ROUGE-L     : {ev['open']['rougeL']:.4f}")
print(f"    Token-F1    : {fmt_ci(ev['open']['f1'], ev['open'].get('ci95_f1'))}")
print(f"  Overall (n={ev['overall']['n']}):")
print(f"    Exact Match : {fmt_ci(ev['overall']['exact_match'], ev['overall'].get('ci95'))}")

## 5. Paired statistical comparison: LoRA vs. zero-shot baseline

This is the **central finding** that the project's RQ1 hangs on. We use:

* **Paired bootstrap** for the difference in overall Exact Match
* **McNemar's exact test** for the binary correctness pattern

In [ ]:
from src.evaluation.statistical_tests import paired_bootstrap_ci_diff, mcnemar_test

# Load both score files
with open('results/baseline/per_example_scores.json') as f:
    base = json.load(f)
with open('checkpoints/lora_quick/per_example_scores.json') as f:
    lora = json.load(f)

# Sanity check: same number of examples in same order
assert base['ids'] == lora['ids'], (
    'Per-example score files are misaligned. Re-run baseline + LoRA evaluation.'
)

diff = paired_bootstrap_ci_diff(base['correct'], lora['correct'], seed=42)
mc   = mcnemar_test(base['correct'], lora['correct'])

print('Δ Overall Exact Match (LoRA − Baseline):')
print(f"  point  : {diff['point']:+.4f} ({100*diff['point']:+.2f} pp)")
print(f"  95% CI : [{diff['lower']:+.4f}, {diff['upper']:+.4f}]")
print(f"  p-value: {diff['p_two_sided']:.4f}  (paired bootstrap)")
print()
print(f"McNemar exact test:")
print(f"  baseline-only correct : b = {mc['b']}")
print(f"  LoRA-only correct     : c = {mc['c']}")
print(f"  p-value               : {mc['p_value']:.4g}")

## 6. Side-by-side comparison table

In [ ]:
with open('results/baseline/metrics.json') as f:
    base_m = json.load(f)

rows = [
    {
        'Method': 'Baseline (zero-shot)',
        'Closed EM': fmt_ci(base_m['closed']['exact_match'], base_m['closed'].get('ci95')),
        'Open F1': fmt_ci(base_m['open']['f1'], base_m['open'].get('ci95_f1')),
        'Overall EM': fmt_ci(base_m['overall']['exact_match'], base_m['overall'].get('ci95')),
        'Trainable params': '0',
        'Train time (s)': '—',
        'Peak GPU (GB)': f"{base_m['meta']['peak_gpu_memory_gb']:.2f}",
    },
    {
        'Method': f"LoRA quick (r={cfg.lora_r}, n_train={cfg.train_max_examples})",
        'Closed EM': fmt_ci(ev['closed']['exact_match'], ev['closed'].get('ci95')),
        'Open F1': fmt_ci(ev['open']['f1'], ev['open'].get('ci95_f1')),
        'Overall EM': fmt_ci(ev['overall']['exact_match'], ev['overall'].get('ci95')),
        'Trainable params': f"{m['params']['trainable']:,}",
        'Train time (s)': f"{m['training']['total_seconds']:.0f}",
        'Peak GPU (GB)': f"{m['peak_gpu_memory_gb']:.2f}",
    },
]
summary = pd.DataFrame(rows)
summary

In [ ]:
# Save the comparison table for the final report
summary.to_csv('checkpoints/lora_quick/comparison_table.csv', index=False)
print('Saved → checkpoints/lora_quick/comparison_table.csv')

## 7. What's next

If this quick run produces sensible numbers (LoRA overall EM ≥ baseline overall EM, with a positive paired-bootstrap CI lower bound), then the pipeline is validated. Members 2 and 3 should:

1. **Member 2 (Weeks 5–6):**
   * Switch `configs/lora_full.yaml` → train on the full 1,797 examples for 3 epochs.
   * For QLoRA: copy `lora_quick.yaml` to `qlora_quick.yaml` and set `load_in_4bit: true`. No code changes needed.
   * Sweep `lora_r ∈ {4, 8, 16, 32}` for the ablation.
2. **Member 3 (Weeks 5–8):**
   * Copy `lora_quick.yaml` to `dora_quick.yaml`, set `use_dora: true`. No code changes needed.
   * Run the same paired statistical comparison as in §5 to compare DoRA vs. baseline and DoRA vs. LoRA.

See `docs/EXTENDING_TO_QLORA_DORA.md` for the full hand-off instructions.